# サンプルデータ前処理

牛サンプル表（血液・骨格筋・Sheet1育種価・高知大母・NLBC母系・種雄牛）を統合し、
`processed/sample_YYYYMMDD/` に parquet / xlsx / HTML / manifest で出力する。

処理本体は **`src/sampledata/sample_pipeline.py`** に関数化してある。
このノートは各関数を順に呼ぶだけ（1関数=1処理）。ロジックを直したいときは `.py` 側を編集する。

> 元の逐次版ノートは `scratch/sampledataの前処理/test.ipynb`（原本として保持）。

## セットアップ

In [ ]:
import importlib
import sys

sys.path.insert(0, "/home/s_mori/JBRT/JBRT_share/src/sampledata")
import sample_pipeline as sp
importlib.reload(sp)   # .py を編集したら再読込（カーネル再起動不要）

## ① ロード

- `build_truth_table`: 正解表（通し番号 ⇔ snp_id）＋DNA濃度
- `load_blood`: 旧血液Excel（接頭辞付き・全列）

In [13]:
truth = sp.build_truth_table()
blood = sp.load_blood()


──── 正解表ビルド ────────────────────────────
   1,488行 / バッチ 10種
   高知大学仔牛 KC-付与 22件 / DNA濃度付与 576件

──── 血液サンプル読込 ────────────────────────
   433行 × 61列 / 9シート
   ラック接頭辞 312件 + シート接頭辞KC- 22件


## ② 縦結合・整形

血液＋骨格筋＋Sheet1 を縦結合してリネーム → 値クリーニング → 型キャスト → 既知誤記の補正。

In [14]:
combined = sp.combine_sources(blood)
combined = sp.clean_values(combined)
combined = sp.cast_types(combined)
combined = sp.apply_corrections(combined)


──── 縦結合＋リネーム ────────────────────────
   8,657行 × 65列 (サンプル 1,433 + Sheet1 7,224)

──── 値クリーニング ──────────────────────────
   specimen/raising英語化・農家名正規化・img文字→NaN・bv欠損記号→NaN

──── 型キャスト ──────────────────────────────
   ID→10桁文字列 / 日付→datetime / Int64 / float / string

──── 既知誤記の補正 ──────────────────────────
   正しい値で上書き修正: 1個体


## ③ Sheet1・高知大母マージ

Sheet1名号の一意化 → Sheet1育種価を cat_id で肉付け → 高知大母リスト（snp_id補完・新規母行・父母名）。

In [15]:
combined = sp.uniquify_sheet1_names(combined)
combined = sp.merge_sheet1(combined)
combined = sp.merge_kochi_dam_snp(combined)
combined = sp.fill_kochi_dam_parents(combined)


──── Sheet1名号の一意化 ──────────────────────
   重複 1,935行 に _N を付与

──── Sheet1マージ ────────────────────────────
   サンプル 1,433行 + Sheet1残 6,858行 = 8,291行
   重複Sheet1行を 366件マージ削除 / 衝突 0
   同一cat_id内 name補完: 1件

──── 高知大母マージ(snp_id/新規行) ───────────
   snp_id補完 22件 / 食い違い 0件
   高知大 母牛を新規追加 5行 → 全体 8,296行

──── 高知大母マージ(父母名) ──────────────────
   父→p1_name補完 7 / 母→m1_name付与 33


## ④ snp_id 補正

正解表 `truth` を使い、通し番号（sample_name）で snp_id を補正・補完。

In [16]:
combined = sp.fix_snp_id(combined, truth)


──── snp_id補正 ──────────────────────────────
   上書き(補正) 253件 / 補完 566件 / 曖昧(未修正) 0件
   補正後: 非欠損 1,359件 / 重複 0件


## ⑤ 血統リンク

登録番号(breed_id)付与 → 父の個体番号(p1_cat_id)解決 → 未登録の種雄牛/父を行追加 →
血統ファイルの名号で父方・母方の親名を backfill（家系図を name で辿れるように）。

In [17]:
combined = sp.add_breed_id(combined)
combined = sp.resolve_p1_cat_id(combined)
combined = sp.add_missing_individuals(combined)
combined = sp.backfill_pedigree_names(combined)


──── breed_id付与 ────────────────────────────
   名号一致で付与: 12件

──── p1_cat_id解決 ───────────────────────────
   解決 1,201件（breed_id法 916 + name法 285）

──── 未登録個体の追加 ────────────────────────
   新規行追加 240件 → 8,536行 × 68列

──── 血統名号のbackfill ──────────────────────
   父方＋母方: p1_name 23件 / m1_name 18件 補完


## ⑥ NLBC スクレイプ（inline・レジューム可）

cat_id を起点に NLBC を母系が尽きるまで再帰検索し、`raw/nlbc_data/nlbc_main.csv`・`nlbc_log.csv` に追記する。

> ⚠ **初回フル実行は母系込みで数千〜1万個体規模＝数時間**（素の Firefox）。チェックポイント＆レジューム付きなので、
> 中断しても同セル再実行で続きから走る。既に全件取得済みなら数十秒で終わる（既検索cat_idはスキップ）。

In [18]:
# 既定 wait=2.0 秒/個体。偽失敗が多いときは wait を増やす（例: sp.scrape_nlbc(combined, wait=3.0)）
nlbc_main, nlbc_log = sp.scrape_nlbc(combined)

display(nlbc_main.head(3)) 
display(nlbc_log.head(3)) 


──── NLBCスクレイプ ──────────────────────────
   起点 cat_id 3,451個体 → 検索開始 [wait=2.0 batch_save=30]

[全世代完了] 個体 4285件 / 異動 32712件 / 母記載 3359件
  失敗(not_found/error) 11件: 例 ['0107101693', '1139972084', '1140021016', '1140032036', '1140244224', '1144788106', '1187348178', '1199566575', '1240344709', '1246505575']
   取得: 個体 4,285件 / 異動 32,712件 / 母記載 3,359件


,cat_id,birth,sex,m1_cat_id,kind,status,error
0,0245030565,2011.07.26,メス,1140161125,褐毛和種,ok,NaN
1,0245030589,2011.08.04,メス,1140212674,褐毛和種,ok,NaN
2,0245030640,2010.07.09,メス,1230606246,褐毛和種,ok,NaN


,cat_id,異動内容,異動年月日,都道府県,市区町村,名称
0,0245030565,出生,2011.07.26,高知県,土佐郡土佐町,窪内 弘幸
1,0245030565,転出,2019.03.05,高知県,土佐郡土佐町,窪内 弘幸
2,0245030565,転入,2019.03.05,高知県,土佐郡土佐町,（株）れいほく畜産


## ⑦ NLBC 統合

NLBC個体情報を combined に統合: birth/death/sex 補完（衝突はNLBC優先）→ age再計算 →
辿った祖先を血縁ノード行として追加 → 無名の母系行に名前/父を補完 → m1_name（母名号）を導出。

In [19]:
combined = sp.apply_nlbc_attributes(combined)
combined = sp.recalc_age(combined)
combined = sp.add_nlbc_ancestors(combined)
combined = sp.backfill_maternal_ancestors(combined)
combined = sp.derive_m1_name(combined)


──── NLBC属性の統合 ──────────────────────────
   birth: 衝突 2件 → NLBC優先で上書き
          cat_id    name         既存       NLBC
      1679432703    <NA> 2023-08-02 2023-08-12
      1487522580 第８８ことぶき 2014-03-30 2016-03-30
   birth: 補完 2,100 + 上書き 2 = 計 2,102行変更
   death: 衝突 15件 → NLBC優先で上書き
          cat_id    name         既存       NLBC
      1253491751 第２のぶたか１ 2013-07-20 2018-02-01
      1144805056   さちこ_7 2020-10-06 2023-06-23
      1180997098  第２７いちこ 2021-07-21 2021-07-06
      1185369128  あつひめ_1 2019-02-25 2019-02-05
      1240301353    嶺北五月 2014-01-19 2024-01-19
      1182440288 第２５はつしげ 2009-05-21 2019-05-21
      1246831601   もとさかえ 2018-11-23 2018-11-13
      1655584617    <NA> 2025-01-30 2025-01-31
      1458002820    <NA> 2022-02-25 2025-02-25
      1660300127    <NA> 2025-05-23 2025-05-26
      1656451901    <NA> 2025-06-17 2025-06-19
      1663907323    <NA> 2025-06-19 2025-06-20
      1482551684    <NA> 2025-06-20 2025-06-23
      1671602388    <NA> 2025-06-23 2025-06-24
      167965

## ⑧ フラグ・グループ・整列・保存

has_snp/dupli_snp 付与 → group（複数メンバーシップ）付与 → COLUMN_ORDER で整列（ここで1回だけ）→ 出力。

In [20]:
combined = sp.add_snp_flags(combined)
combined = sp.assign_group(combined)
combined = sp.reorder_columns(combined)          # 整列は末尾で1回だけ
combined = sp.save_outputs(combined, version="20260708")   # processed/sample_<version>/ に出力


──── SNPフラグ付与 ───────────────────────────
   has_snp 1,348件（snp_idありだがgenotype無し 11件）
   dupli_snp 28行 / 14個体（同一個体で複数genotype）

──── group付与 ───────────────────────────────
   横断メンバーシップ: Sheet1育種価 7,264 / 高知大母 33 / 父 271個体

──── 列並び替え ──────────────────────────────
   9,364行 × 73列

──── 保存 ────────────────────────────────────
   /home/s_mori/JBRT/JBRT_data/processed/sample_20260708
   parquet / xlsx / html / manifest.json  (9,364行 × 73列)
   group(メンバーシップ): {'血液': 433, '骨格筋': 1000, 'Sheet1育種価': 7264, '高知大母': 33, 'NLBC母系': 828, '父(種雄牛)': 271}
